In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Load Data

In [3]:
demographics = pd.read_csv('data/demographics.csv')
current_commitments = pd.read_csv('data/current_commitments.csv')
prior_commitments = pd.read_csv('data/prior_commitments.csv')
    
print(f"Demographics shape: {demographics.shape}")
print(f"Current commitments shape: {current_commitments.shape}")
print(f"Prior commitments shape: {prior_commitments.shape}")

/var/folders/qq/2xq_hbb10j75t18jqwpwnfzh0000gn/T/ipykernel_10038/363343874.py:2: DtypeWarning: Columns (7,22,23,24,25,26,27,28,29,30,31,32,33) have mixed types. Specify dtype option on import or set low_memory=False.
  current_commitments = pd.read_csv('data/current_commitments.csv')


Demographics shape: (95476, 13)
Current commitments shape: (369125, 34)
Prior commitments shape: (191436, 13)


# Understanding the Data

## Demographics

In [5]:
# First 5 Rows
demographics.head()

,CDCNo,Ethnicity,Controlling Offense,Description,Offense Begin Date,Offense End Date,Controlling Case Number,Controlling Case Sentencing County,Sentence Type,Aggregate Sentence in Months,Offense Category,EPRD/MEPD Month and Year,Current Location
0,2cf2a233c4,Black,VC10851(a),Vehicle Theft,2022-12-07,2022-12-07,FVI22003547,San Bernardino,Second Striker,32,Property Crimes,APR24,Central California Women's Facility
1,5a72696541,White,PC187 2nd,Murder 2nd,2012-09-18,2012-09-18,12F06402,Sacramento,Life with Parole,360,Crimes Against Persons,NOV33,Central California Women's Facility
2,7d608b6a4c,White,PC187 2nd,Murder 2nd,2010-05-18,2010-05-18,CM032513,Butte,Life with Parole,300,Crimes Against Persons,SEP28,Central California Women's Facility
3,39c1bc8c2f,Other,PC459,Burglary 1st,1998-01-20,1998-01-20,CM010387,Butte,Third Striker,348,Property Crimes,NOV22,California Institution for Women
4,220f2cdfc5,Black,PC187,Murder 1st,2017-03-21,2017-03-21,BA455966,Los Angeles,Life with Parole,312,Crimes Against Persons,MAY35,California Institution for Women


### Observations from first 5 rows:
- `CDCNo` is unique id, can merge datasets on this value
- `Controlling Offense` column might need to be encoded
- Columns to probably keep: `CDCNo`, `Ethnicity`,`Description`, `Sentence Type`, `Aggregate Sentence in Months`, `Offense Category` 
- Columns unsure about keeping: `Controlling Offense`, `EPRD/MEPD Month and Year`
- Columns to drop: `Offense Begin Date`, `Offense End Date`, `Controlling Case Number`, `Current Location`

### Visualization ideas from first 5 rows
- Pie Chart on `Ethnicity`
- `Offense Category` vs. `Aggrefate Sentence in Months`

In [8]:
# Missing values
demographics.isna().sum()

CDCNo                                     0
Ethnicity                                 0
Controlling Offense                       1
Description                               1
Offense Begin Date                        1
Offense End Date                          1
Controlling Case Number               25356
Controlling Case Sentencing County        0
Sentence Type                             0
Aggregate Sentence in Months              0
Offense Category                          0
EPRD/MEPD Month and Year                  1
Current Location                          0
dtype: int64

In [10]:
# Unique Values
demographics_columns = ["Ethnicity", "Controlling Offense", "Description", "Sentence Type", "Offense Category"]

for col in demographics_columns:
    print(f"Number of unique values in {col}: {demographics[col].nunique()}")

Number of unique values in Ethnicity: 28
Number of unique values in Controlling Offense: 483
Number of unique values in Description: 528
Number of unique values in Sentence Type: 7
Number of unique values in Offense Category: 4


In [34]:
demographics["Sentence Type"].unique()

array(['Second Striker', 'Life with Parole', 'Third Striker', 'DSL',
       'Life w/o Parole', 'Condemned', 'Other'], dtype=object)

### Notes From Unique Values:
- Might only use `Ethnicity`, `Sentency Type`, and `Offense Category` to simplify the visuals for now

## Current Commitments

In [13]:
current_commitments.columns

Index(['CDCNo', 'Sentencing County', 'Case Number',
       'Sentence From Abstract of Judgement', 'Offense', 'Offense Description',
       'Offense Category', 'In-prison', 'Offense Begin Date',
       'Offense End Date', 'Offense Time with Enhancement', 'Relationship',
       'Off_Enh1', 'Off_Enh_Desc1', 'Off_Enh2', 'Off_Enh_Desc2', 'Off_Enh3',
       'Off_Enh_Desc3', 'Off_Enh4', 'Off_Enh_Desc4', 'Off_Enh5',
       'Off_Enh_Desc5', 'Off_Enh6', 'Off_Enh_Desc6', 'Off_Enh7',
       'Off_Enh_Desc7', 'Off_Enh8', 'Off_Enh_Desc8', 'Off_Enh9',
       'Off_Enh_Desc9', 'Off_Enh10', 'Off_Enh_Desc10', 'Off_Enh11',
       'Off_Enh_Desc11'],
      dtype='object')

### Notes About the Columns:
- `Sentence From Abstract of Judgement`: Sentence length in text form, will need to be encoded
- `Off_Enh**` -- ??

In [15]:
current_commitments.head()

,CDCNo,Sentencing County,Case Number,Sentence From Abstract of Judgement,Offense,Offense Description,Offense Category,In-prison,Offense Begin Date,Offense End Date,...,Off_Enh7,Off_Enh_Desc7,Off_Enh8,Off_Enh_Desc8,Off_Enh9,Off_Enh_Desc9,Off_Enh10,Off_Enh_Desc10,Off_Enh11,Off_Enh_Desc11
0,2cf2a233c4,San Bernardino,FVI22003547,2 Years 8 Months,VC10851(a),Vehicle Theft,Property Crimes,NaN,2022-12-07,2022-12-07 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5a72696541,Sacramento,12F06402,Life with Parole,PC191.5(c)(2),Vehicular Manslaughter While Intoxicated,Crimes Against Persons,NaN,2012-09-18,2012-09-18 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5a72696541,Sacramento,12F06402,Life with Parole,PC187 2nd,Murder 2nd,Crimes Against Persons,NaN,2012-09-18,2012-09-18 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7d608b6a4c,Butte,CM032513,Life with Parole,PC187 2nd,Murder 2nd,Crimes Against Persons,NaN,2010-05-18,2010-05-18 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,39c1bc8c2f,San Bernardino,FWV1404908,1 Years 4 Months,PC4573.8,Possession Paraphernalia/Drugs/Alcohol in Jail...,Drug Crimes,In-Prison,2014-09-12,2014-09-12 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Notes From First 5 Rows:
- Columns to keep: `CDCNo`, `Sentencing County`, `Offense Category`
- Columns to maybe keep: `Case Number`, `Sentence From Abstract of Judgement`, `Offense`, `Offense Description`
- Drop everything else

In [16]:
# Missing values
current_commitments.isna().sum()

CDCNo                                       0
Sentencing County                           1
Case Number                            133276
Sentence From Abstract of Judgement        61
Offense                                     1
Offense Description                         1
Offense Category                            0
In-prison                              352794
Offense Begin Date                      37455
Offense End Date                        37455
Offense Time with Enhancement               0
Relationship                                1
Off_Enh1                               273164
Off_Enh_Desc1                          273164
Off_Enh2                               347518
Off_Enh_Desc2                          347518
Off_Enh3                               364226
Off_Enh_Desc3                          364226
Off_Enh4                               367665
Off_Enh_Desc4                          367665
Off_Enh5                               368715
Off_Enh_Desc5                     

In [22]:
current_commitments_columns = ["Sentencing County", "Sentence From Abstract of Judgement", "Offense", "Offense Description", "Offense Category", "Offense Time with Enhancement", "Relationship"]

for col in current_commitments_columns:
    print(f"Number of unique values in {col}: {current_commitments[col].nunique()}")

Number of unique values in Sentencing County: 57
Number of unique values in Sentence From Abstract of Judgement: 506
Number of unique values in Offense: 835
Number of unique values in Offense Description: 890
Number of unique values in Offense Category: 5
Number of unique values in Offense Time with Enhancement: 324
Number of unique values in Relationship: 4


## Prior Commitments

In [23]:
prior_commitments.head()

,CDCNo,Sentencing County,Case Number,Sentence From Abstract of Judgement,Offense,Offense Description,Offense Category,In-prison,Offense Begin Date,Offense End Date,Offense Time with Enhancement,Relationship,Release Date
0,2cf2a233c4,Los Angeles,KA048775,1 Years 4 Months,HS11350(a),Possess Controlled Substance,Drug Crimes,NaN,2000-06-06,2000-06-06,1 Year 4 Months,Concurrent,2001-03-08
1,2cf2a233c4,Los Angeles,KA035079,2 Years 8 Months,PC459 2nd,Burglary 2nd,Property Crimes,NaN,1996-11-25,1996-11-25,2 Years 0 Months,Initial,1998-08-14
2,2cf2a233c4,Los Angeles,KA020374,3 Years,HS11352(a),Transport/Sell Controlled Substance,Drug Crimes,NaN,1993-12-10,1993-12-10,3 Years 0 Months,Initial,1995-09-18
3,2cf2a233c4,San Bernardino,FSB1001777,4 Years,PC667.5(b),Prior Prison Term/Non Violent new offense is a...,Case Enhancement,NaN,NaN,NaN,1 Year,Consecutive,NaN
4,2cf2a233c4,Los Angeles,KA035079,2 Years 8 Months,PC666,Petty Theft With Prior,Property Crimes,NaN,1996-12-17,1996-12-17,2 Years 0 Months,Concurrent,1998-08-14


In [26]:
prior_commitments.isna().sum()

CDCNo                                       0
Sentencing County                           1
Case Number                             35880
Sentence From Abstract of Judgement      1373
Offense                                     1
Offense Description                         1
Offense Category                            0
In-prison                              188276
Offense Begin Date                      22188
Offense End Date                        22188
Offense Time with Enhancement               0
Relationship                                1
Release Date                            22083
dtype: int64

# Data Cleaning

## Demographics

In [30]:
# Convert date columns to datetime
demographics['Offense Begin Date'] = pd.to_datetime(demographics['Offense Begin Date'], errors='coerce')
demographics['Offense End Date'] = pd.to_datetime(demographics['Offense End Date'], errors='coerce')

demographics[['Offense Begin Date', 'Offense End Date']].head()

,Offense Begin Date,Offense End Date
0,2022-12-07,2022-12-07
1,2012-09-18,2012-09-18
2,2010-05-18,2010-05-18
3,1998-01-20,1998-01-20
4,2017-03-21,2017-03-21
